# Thực hành Chương 2 — Single-Agent Reinforcement Learning
## Bài 2: SARSA với môi trường FrozenLake & So sánh với Q-Learning

**Họ tên:** ___________________________  
**MSSV:** ___________________________  
**Ngày thực hành:** ___________________________

---

## 1. Giới thiệu SARSA

**SARSA** là thuật toán học tăng cường **on-policy** — tên gọi xuất phát từ 5 phần tử trong mỗi bước cập nhật:

$$(S, A, R, S', A')$$

### Công thức cập nhật SARSA:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma Q(s', a') - Q(s, a) \right]$$

### So sánh nhanh với Q-Learning:

| Tiêu chí | Q-Learning | SARSA |
|---|---|---|
| Loại | Off-policy | On-policy |
| Cập nhật theo | $\max_{a'} Q(s', a')$ (hành động tốt nhất) | $Q(s', a')$ (hành động thực tế chọn) |
| Đặc điểm | Học aggressively | Học conservatively, an toàn hơn |
| Phù hợp | Môi trường ít rủi ro | Môi trường có nhiều trạng thái nguy hiểm |

## 2. Cài đặt thư viện

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

print("Phiên bản thư viện:")
print(f"  numpy     : {np.__version__}")
print(f"  gymnasium : {gym.__version__}")

## 3. Hàm dùng chung cho cả hai thuật toán

In [ ]:
def choose_action(state, q_table, epsilon, env):
    """
    Chính sách ε-greedy:
    - Xác suất epsilon   : hành động ngẫu nhiên (exploration)
    - Xác suất 1-epsilon : hành động tốt nhất theo Q-table (exploitation)
    """
    if np.random.rand() < epsilon:
        return env.action_space.sample()   # Exploration
    else:
        return np.argmax(q_table[state])   # Exploitation


def moving_average(data, window=200):
    """Tính trung bình trượt để làm mượt đường cong học."""
    return np.convolve(data, np.ones(window) / window, mode='valid')


print("Các hàm dùng chung đã định nghĩa.")

## 4. Thiết lập siêu tham số chung

In [ ]:
# -----------------------------------------------
# Siêu tham số — dùng chung cho Q-Learning & SARSA
# -----------------------------------------------
ALPHA         = 0.8      # Learning rate
GAMMA         = 0.95     # Discount factor
EPSILON       = 1.0      # Epsilon khởi tạo
EPSILON_MIN   = 0.01     # Epsilon tối thiểu
EPSILON_DECAY = 0.995    # Tốc độ giảm epsilon
N_EPISODES    = 10_000   # Số episode huấn luyện

print("Siêu tham số:")
print(f"  Alpha         : {ALPHA}")
print(f"  Gamma         : {GAMMA}")
print(f"  Epsilon decay : {EPSILON_DECAY}")
print(f"  Số episode    : {N_EPISODES}")

## 5. Hàm huấn luyện Q-Learning

In [ ]:
def train_q_learning(env, n_episodes, alpha, gamma, epsilon, epsilon_min, epsilon_decay):
    """
    Huấn luyện Q-Learning (off-policy).

    Cập nhật: Q(s,a) += alpha * [r + gamma * max Q(s',a') - Q(s,a)]

    Trả về:
        q_table          : Q-table sau huấn luyện
        rewards_per_ep   : danh sách tổng reward mỗi episode
    """
    n_states  = env.observation_space.n
    n_actions = env.action_space.n
    q_table   = np.zeros((n_states, n_actions))
    rewards_per_ep = []
    eps = epsilon

    for episode in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0
        done = False

        while not done:
            # Chọn hành động theo ε-greedy
            action = choose_action(state, q_table, eps, env)

            # Thực hiện hành động
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # Cập nhật Q-Learning: dùng max Q(s', a') — off-policy
            best_next_q = np.max(q_table[next_state])
            td_target   = reward + gamma * best_next_q
            td_error    = td_target - q_table[state, action]
            q_table[state, action] += alpha * td_error

            state = next_state
            total_reward += reward

        rewards_per_ep.append(total_reward)
        eps = max(epsilon_min, eps * epsilon_decay)

        if (episode + 1) % 2000 == 0:
            avg = np.mean(rewards_per_ep[-1000:])
            print(f"  [Q-Learning] Episode {episode + 1:6d} | Avg Reward: {avg:.3f} | ε: {eps:.3f}")

    return q_table, rewards_per_ep

print("Hàm train_q_learning đã định nghĩa.")

## 6. Hàm huấn luyện SARSA

In [ ]:
def train_sarsa(env, n_episodes, alpha, gamma, epsilon, epsilon_min, epsilon_decay):
    """
    Huấn luyện SARSA (on-policy).

    Cập nhật: Q(s,a) += alpha * [r + gamma * Q(s',a') - Q(s,a)]
    Khác với Q-Learning: a' được chọn THỰC SỰ theo ε-greedy (không dùng max).

    Trả về:
        q_table          : Q-table sau huấn luyện
        rewards_per_ep   : danh sách tổng reward mỗi episode
    """
    n_states  = env.observation_space.n
    n_actions = env.action_space.n
    q_table   = np.zeros((n_states, n_actions))
    rewards_per_ep = []
    eps = epsilon

    for episode in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0
        done = False

        # SARSA: chọn hành động đầu tiên TRƯỚC khi vào vòng lặp
        action = choose_action(state, q_table, eps, env)

        while not done:
            # Thực hiện hành động a
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # Chọn hành động tiếp theo a' theo ε-greedy (on-policy)
            next_action = choose_action(next_state, q_table, eps, env)

            # Cập nhật SARSA: dùng Q(s', a') thực tế — on-policy
            td_target = reward + gamma * q_table[next_state, next_action]
            td_error  = td_target - q_table[state, action]
            q_table[state, action] += alpha * td_error

            # Chuyển sang (s', a')
            state  = next_state
            action = next_action
            total_reward += reward

        rewards_per_ep.append(total_reward)
        eps = max(epsilon_min, eps * epsilon_decay)

        if (episode + 1) % 2000 == 0:
            avg = np.mean(rewards_per_ep[-1000:])
            print(f"  [SARSA]      Episode {episode + 1:6d} | Avg Reward: {avg:.3f} | ε: {eps:.3f}")

    return q_table, rewards_per_ep

print("Hàm train_sarsa đã định nghĩa.")

## 7. Huấn luyện cả hai thuật toán

In [ ]:
# -----------------------------------------------
# Tạo môi trường FrozenLake (is_slippery=True)
# -----------------------------------------------
env = gym.make("FrozenLake-v1", is_slippery=True)

# --- Huấn luyện Q-Learning ---
print("=" * 55)
print("Bắt đầu huấn luyện Q-Learning ...")
q_table_ql, rewards_ql = train_q_learning(
    env, N_EPISODES, ALPHA, GAMMA,
    EPSILON, EPSILON_MIN, EPSILON_DECAY
)
print("Q-Learning hoàn tất!")

# --- Huấn luyện SARSA ---
print("=" * 55)
print("Bắt đầu huấn luyện SARSA ...")
q_table_sarsa, rewards_sarsa = train_sarsa(
    env, N_EPISODES, ALPHA, GAMMA,
    EPSILON, EPSILON_MIN, EPSILON_DECAY
)
print("SARSA hoàn tất!")
print("=" * 55)

## 8. So sánh Learning Curve: Q-Learning vs SARSA

In [ ]:
# -----------------------------------------------
# Vẽ learning curve của cả hai thuật toán trên cùng biểu đồ
# -----------------------------------------------
window = 200
smooth_ql    = moving_average(rewards_ql,    window)
smooth_sarsa = moving_average(rewards_sarsa, window)
x_range = range(window - 1, N_EPISODES)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Biểu đồ 1: Reward thô ---
axes[0].plot(rewards_ql,    alpha=0.3, color='steelblue', label='Q-Learning')
axes[0].plot(rewards_sarsa, alpha=0.3, color='tomato',    label='SARSA')
axes[0].set_title('Reward mỗi episode (thô)')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Tổng Reward')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# --- Biểu đồ 2: Moving average ---
axes[1].plot(x_range, smooth_ql,    color='steelblue', linewidth=2, label='Q-Learning')
axes[1].plot(x_range, smooth_sarsa, color='tomato',    linewidth=2, label='SARSA')
axes[1].set_title(f'Reward trung bình trượt (window={window})')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Avg Reward')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('So sánh Q-Learning vs SARSA — FrozenLake (is_slippery=True)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nTóm tắt hiệu suất cuối (1000 episode cuối):")
print(f"  Q-Learning : {np.mean(rewards_ql[-1000:]):.4f}")
print(f"  SARSA      : {np.mean(rewards_sarsa[-1000:]):.4f}")

## 9. Thử nghiệm: Thay đổi Epsilon Decay và phân tích

In [ ]:
# -----------------------------------------------
# So sánh 3 mức epsilon decay khác nhau cho SARSA
# -----------------------------------------------
decay_values = [0.999, 0.995, 0.990]   # Chậm / Trung bình / Nhanh
decay_labels = ['Decay chậm (0.999)', 'Decay trung bình (0.995)', 'Decay nhanh (0.990)']
decay_colors = ['green', 'orange', 'purple']

plt.figure(figsize=(10, 5))

for decay, label, color in zip(decay_values, decay_labels, decay_colors):
    _, rewards = train_sarsa(
        env, N_EPISODES, ALPHA, GAMMA,
        EPSILON, EPSILON_MIN, decay
    )
    smooth = moving_average(rewards, window=200)
    plt.plot(range(199, N_EPISODES), smooth,
             color=color, linewidth=2, label=label)
    print(f"  {label}: avg reward cuối = {np.mean(rewards[-1000:]):.4f}")

plt.title('SARSA — Ảnh hưởng của Epsilon Decay')
plt.xlabel('Episode')
plt.ylabel('Avg Reward (moving avg)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Đánh giá agent sau huấn luyện

In [ ]:
def evaluate(q_table, env, n_eval=1000):
    """
    Đánh giá agent bằng cách chạy n_eval episodes với epsilon = 0
    (luôn chọn hành động tốt nhất theo Q-table).

    Trả về:
        win_rate : tỉ lệ thắng (%)
    """
    wins = 0
    for _ in range(n_eval):
        state, _ = env.reset()
        done = False
        while not done:
            action = np.argmax(q_table[state])  # Greedy (epsilon = 0)
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
        if reward == 1.0:
            wins += 1
    return wins / n_eval * 100


# --- Đánh giá cả hai thuật toán ---
N_EVAL = 1000
win_ql    = evaluate(q_table_ql,    env, N_EVAL)
win_sarsa = evaluate(q_table_sarsa, env, N_EVAL)

print(f"Kết quả đánh giá trên {N_EVAL} episodes (epsilon = 0):")
print(f"  Q-Learning : {win_ql:.1f}% thắng")
print(f"  SARSA      : {win_sarsa:.1f}% thắng")

## 11. Trả lời câu hỏi lý thuyết

### Câu 2: So sánh công thức cập nhật Q-value

| | Q-Learning | SARSA |
|---|---|---|
| **Công thức** | $Q(s,a) \leftarrow Q(s,a) + \alpha[r + \gamma \max_{a'} Q(s', a') - Q(s,a)]$ | $Q(s,a) \leftarrow Q(s,a) + \alpha[r + \gamma Q(s', a') - Q(s,a)]$ |
| **Điểm khác biệt** | Dùng $\max Q(s', a')$ — hành động **tốt nhất giả định** | Dùng $Q(s', a')$ — hành động **thực tế được chọn** |
| **Loại** | Off-policy | On-policy |

**Ảnh hưởng đến chiến lược học:**

> *(Sinh viên điền câu trả lời vào đây)*
>
> Q-Learning luôn cập nhật theo kịch bản lạc quan nhất (hành động tốt nhất), nên học nhanh hơn về giá trị lý thuyết nhưng có thể không phản ánh đúng rủi ro thực tế khi exploration vẫn đang cao.
>
> SARSA cập nhật theo đúng hành động mà agent sẽ thực sự thực hiện (bao gồm cả hành động ngẫu nhiên do ε-greedy), nên Q-value học được phản ánh rủi ro thực tế hơn.

---

### Câu 4: Trong môi trường rủi ro cao, thuật toán nào an toàn hơn?

> *(Sinh viên điền câu trả lời vào đây)*
>
> **SARSA an toàn hơn** trong môi trường nhiều trạng thái nguy hiểm.
>
> Lý do: SARSA cập nhật Q-value dựa trên hành động thực tế chọn theo ε-greedy — bao gồm cả các hành động ngẫu nhiên. Khi ε còn lớn, SARSA sẽ "tính đến" xác suất chọn nhầm hành động và đi vào trạng thái nguy hiểm, dẫn đến Q-value của các trạng thái gần nguy hiểm thấp hơn → agent học cách tránh xa khu vực đó.
>
> Q-Learning thì không quan tâm đến hành động thực tế: nó luôn tính theo $\max Q(s', a')$ nên dễ đánh giá quá cao giá trị của trạng thái gần nguy hiểm → dễ bị té ngã hơn khi exploration còn cao.

---

### Câu 5: Nguyên nhân reward dao động mạnh / không tăng

| # | Nguyên nhân | Cách khắc phục |
|---|---|---|
| 1 | **ε decay quá chậm**: agent vẫn khám phá nhiều dù đã học đủ | Tăng tốc epsilon decay (ví dụ: 0.995 → 0.990) |
| 2 | **Learning rate (α) quá lớn**: Q-value dao động không ổn định | Giảm alpha (ví dụ: từ 0.8 xuống 0.3–0.5) |
| 3 | **Số episode quá ít**: chưa đủ dữ liệu để Q-table hội tụ | Tăng số episode huấn luyện (ví dụ: 20,000–50,000) |

---
## Tổng kết Bài 2

| Nội dung | Q-Learning | SARSA |
|---|---|---|
| Loại thuật toán | Off-policy TD | On-policy TD |
| Cập nhật dựa trên | Hành động tối ưu giả định | Hành động thực tế |
| Tốc độ học | Nhanh hơn | Chậm hơn nhưng ổn định |
| Độ an toàn | Thấp hơn ở môi trường rủi ro | Cao hơn |

**Kết luận:**  
- **Q-Learning** phù hợp khi cần tối ưu hóa tối đa hiệu suất trong môi trường ít rủi ro.  
- **SARSA** phù hợp hơn khi môi trường có nhiều trạng thái nguy hiểm vì nó học chính sách an toàn hơn trong giai đoạn exploration.  
- Epsilon decay ảnh hưởng quan trọng đến tốc độ và chất lượng hội tụ: cần điều chỉnh phù hợp với từng bài toán.